<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/sistema_graphrag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema GraphRAG — Leyes de inversión (Neo4j + OpenAI)

- **Setup** (compartido): dependencias, credenciales, config, driver, y objetos compartidos (embedder, LLM, prompt).
- **Parte A — Backfill** (correr *una sola vez*): embeddings + índices.
- **Parte B — Retriever GraphRAG** (uso normal): recuperación con expansión al grafo + pipeline.
- **Parte C — Baseline y comparación**: mismo sistema con el grafo apagado.
- **Parte D — Conversación con memoria** (LangGraph, Etapa 1): chat multi-turno que reusa el pipeline.

**Antes de correr:** en **Colab → Secrets** cargá (con acceso a este notebook): `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.

**Orden:** Setup → (Parte A si hace falta) → Parte B → Parte C / Parte D.

## Setup (compartido)

Todas las dependencias se instalan en una sola celda (evita choques de versión). El `embedder`, el `llm` y el `prompt_template` se definen acá para que los usen las Partes B, C y D.

> ⚠ **Si venís de una corrida con error de imports** (por ejemplo tras haber instalado algo a mitad de sesión), hacé **Runtime → Reiniciar sesión** y volvé a correr desde esta celda. Con este notebook, al instalar todo junto arriba, no debería hacer falta.

In [1]:
# Dependencias (todo en una sola instalacion coherente, para evitar choques de version).
# neo4j-graphrag trae el driver de neo4j y openai; langgraph trae langchain-core.
!pip install -q "neo4j_graphrag[openai]" langgraph tiktoken tenacity

In [2]:
# Credenciales desde Colab Secrets (o por teclado fuera de Colab)
import os
_KEYS = ["OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE"]
try:
    from google.colab import userdata
    for k in _KEYS:
        os.environ[k] = userdata.get(k)
    print("Credenciales cargadas desde Colab Secrets.")
except Exception:
    import getpass
    for k in _KEYS:
        if not os.environ.get(k):
            os.environ[k] = getpass.getpass(f"{k}: ")
    print("Credenciales cargadas por teclado.")

Credenciales cargadas desde Colab Secrets.


### Configuración

`EMBEDDING_MODEL` es el mismo para backfill y retriever. `TEST_LIMIT=20` deja la Parte A en modo prueba; poné `None` para el corpus completo.

In [3]:
import os, time
from openai import OpenAI
from neo4j import GraphDatabase
import tiktoken
from tenacity import retry, wait_random_exponential, stop_after_attempt

# (opcional) silenciar el warning de deprecacion de db.index.vector.queryNodes:
# import logging; logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

# --- Conexion ---
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USER     = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]     # tu base: ece63d51

# --- Modelos ---
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
EMBEDDING_MODEL      = "text-embedding-3-small"   # backfill Y retriever: deben coincidir
EMBEDDING_DIM        = 1536
EMBEDDING_ENCODING   = "cl100k_base"
MAX_TOKENS_PER_INPUT = 8000
LLM_MODEL            = "gpt-4o"                    # comparable con ALIBot (Belen)

# --- Esquema del grafo ---
LABEL_ARTICULO    = "Articulo"
LABEL_RECUPERABLE = "Recuperable"
REL_TIENE_ART     = "TIENE_ARTICULO"
NORM_LABELS       = ["Ley", "Decreto", "Resolucion"]
PROP_TEXTO        = "texto"
PROP_NUM_ART      = "numero"
PROP_PARTE        = "parte"
PROP_LEY_NUMERO   = "ley_numero"
PROP_NUM_NORMA    = "numero"
PROP_NOMBRE_COMPLETO = "nombre_completo"
PROP_EMBEDDING    = "embedding"
PROP_ES_PRUEBA    = "es_prueba"

# --- Indices ---
VECTOR_INDEX_NAME   = "recuperable_embedding"
FULLTEXT_ART_NAME   = "articulo_texto_ft"
FULLTEXT_NORMA_NAME = "normas_nombre_ft"

# --- Tunables ---
EMBED_BATCH_SIZE = 50
WRITE_BATCH_SIZE = 500
TEST_LIMIT       = None     # backfill: 20 prueba; None = corpus completo
TOP_K            = 5      # retriever

# --- Clientes y driver COMPARTIDOS ---
client = OpenAI(api_key=OPENAI_API_KEY)          # usado por el backfill
enc    = tiktoken.get_encoding(EMBEDDING_ENCODING)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Setup listo (config + clientes + driver).")

Setup listo (config + clientes + driver).


### Objetos compartidos (embedder, LLM, prompt)

In [4]:
# Objetos COMPARTIDOS: embedder, LLM y prompt.
# Los usan el retriever GraphRAG, el baseline y (mas adelante) LangGraph.
# Definirlos aca, en el Setup, evita que una celda dependa de que otra se haya corrido antes.
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.generation import RagTemplate

embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL)   # MISMO modelo que el backfill
llm = OpenAILLM(model_name=LLM_MODEL, model_params={"temperature": 0})

TEMPLATE = """Sos un asistente legal sobre derecho de inversiones de Paraguay.
Respondé usando SOLO el contexto. Citá la norma y el artículo en cada afirmación.
Si una norma figura como DEROGADA o con estado distinto de vigente, aclaralo
explícitamente y NO la presentes como vigente. Si el contexto no alcanza, decilo.

# Contexto:
{context}
{examples}
# Pregunta:
{query_text}

# Respuesta:"""
prompt_template = RagTemplate(template=TEMPLATE)
print("Embedder, LLM y prompt listos (compartidos).")

Embedder, LLM y prompt listos (compartidos).


## Parte A — Backfill (correr una sola vez)

Crea los índices y genera los embeddings. **Idempotente**: saltea lo ya embebido. Si ya lo corriste antes, podés saltear esta parte.

In [5]:
def crear_indices(driver):
    driver.execute_query(
        f"""
        CREATE VECTOR INDEX {VECTOR_INDEX_NAME} IF NOT EXISTS
        FOR (n:{LABEL_RECUPERABLE}) ON (n.{PROP_EMBEDDING})
        OPTIONS {{ indexConfig: {{
            `vector.dimensions`: {EMBEDDING_DIM},
            `vector.similarity_function`: 'cosine'
        }} }}
        """,
        database_=NEO4J_DATABASE,
    )
    driver.execute_query(
        f"CREATE FULLTEXT INDEX {FULLTEXT_ART_NAME} IF NOT EXISTS "
        f"FOR (a:{LABEL_ARTICULO}) ON EACH [a.{PROP_TEXTO}]",
        database_=NEO4J_DATABASE,
    )
    normas = "|".join(NORM_LABELS)
    driver.execute_query(
        f"CREATE FULLTEXT INDEX {FULLTEXT_NORMA_NAME} IF NOT EXISTS "
        f"FOR (n:{normas}) ON EACH [n.{PROP_NOMBRE_COMPLETO}]",
        database_=NEO4J_DATABASE,
    )
    print("Indices creados/verificados.")


def leer_pendientes(driver):
    norm_pred = " OR ".join(f"n:{l}" for l in NORM_LABELS)
    limit_clause = f"LIMIT {TEST_LIMIT}" if TEST_LIMIT else ""
    query = f"""
        MATCH (a:{LABEL_ARTICULO})
        WHERE a.{PROP_TEXTO} IS NOT NULL AND trim(a.{PROP_TEXTO}) <> ''
              AND a.{PROP_EMBEDDING} IS NULL
              AND coalesce(a.{PROP_ES_PRUEBA}, false) = false
        OPTIONAL MATCH (n)-[:{REL_TIENE_ART}]->(a)
            WHERE {norm_pred}
        WITH a, collect(n)[0] AS norm
        RETURN elementId(a) AS eid,
               a.{PROP_NUM_ART}    AS art_num,
               a.{PROP_PARTE}      AS parte,
               a.{PROP_TEXTO}      AS texto,
               a.{PROP_LEY_NUMERO} AS ley_numero,
               CASE WHEN norm IS NULL THEN null
                    ELSE head([l IN labels(norm) WHERE l IN {NORM_LABELS}]) END AS norm_tipo,
               norm.{PROP_NUM_NORMA} AS norm_num
        {limit_clause}
    """
    records, _, _ = driver.execute_query(query, database_=NEO4J_DATABASE)
    return [r.data() for r in records]


def texto_con_encabezado(row):
    partes = []
    tipo = row.get("norm_tipo")
    num = row.get("norm_num") or row.get("ley_numero")
    if tipo and num:
        partes.append(f'{tipo} {num}')
    elif num:
        partes.append(f'{num}')
    if row.get("art_num") is not None:
        partes.append(f'Artículo {row["art_num"]}')
    if row.get("parte") and row["parte"] not in (None, "", "cuerpo"):
        partes.append(f'({row["parte"]})')
    prefijo = ", ".join(partes)
    return f'{prefijo}: {row["texto"]}' if prefijo else row["texto"]


def recortar_a_limite(texto):
    toks = enc.encode(texto)
    if len(toks) <= MAX_TOKENS_PER_INPUT:
        return texto, len(toks), False
    return enc.decode(toks[:MAX_TOKENS_PER_INPUT]), MAX_TOKENS_PER_INPUT, True


@retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(6))
def embeber_lote(textos):
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=textos)
    return [d.embedding for d in resp.data]


def escribir_vectores(driver, filas):
    query = f"""
        UNWIND $rows AS row
        MATCH (a) WHERE elementId(a) = row.eid
        CALL db.create.setNodeVectorProperty(a, $prop, row.embedding)
        SET a:{LABEL_RECUPERABLE}
    """
    for i in range(0, len(filas), WRITE_BATCH_SIZE):
        lote = filas[i:i + WRITE_BATCH_SIZE]
        driver.execute_query(query, rows=lote, prop=PROP_EMBEDDING, database_=NEO4J_DATABASE)


def verificar(driver):
    q = f"""
        MATCH (a:{LABEL_ARTICULO})
        RETURN count(a) AS total,
               count(a.{PROP_EMBEDDING}) AS con_embedding,
               sum(CASE WHEN a.{PROP_TEXTO} IS NOT NULL
                        AND a.{PROP_EMBEDDING} IS NULL THEN 1 ELSE 0 END) AS pendientes
    """
    records, _, _ = driver.execute_query(q, database_=NEO4J_DATABASE)
    print("Verificacion:", records[0].data())

### Correr el backfill

Primero con `TEST_LIMIT = 20`; verificá y después poné `TEST_LIMIT = None` en la config y volvé a correr la config y esta celda.

In [6]:
# Parte A — correr el backfill (una sola vez). Usa el 'driver' compartido del Setup.
# TEST_LIMIT=20 (config) hace una prueba; pone None para el corpus completo.
crear_indices(driver)
pendientes = leer_pendientes(driver)
print(f"Articulos pendientes de embeber: {len(pendientes)}")
if pendientes:
    total_tokens = 0
    preparados, recortados = [], []
    for row in pendientes:
        texto_final, n_tok, fue_recortado = recortar_a_limite(texto_con_encabezado(row))
        total_tokens += n_tok
        preparados.append((row["eid"], texto_final))
        if fue_recortado:
            recortados.append(row["eid"])
    costo = total_tokens / 1_000_000 * 0.02
    print(f"Tokens aprox: {total_tokens:,} | costo estimado: ~US${costo:.4f}")
    if recortados:
        print(f"AVISO: {len(recortados)} articulos se recortaron: {recortados[:10]}")
    hechos = 0
    for i in range(0, len(preparados), EMBED_BATCH_SIZE):
        lote = preparados[i:i + EMBED_BATCH_SIZE]
        textos = [t for _, t in lote]
        vectores = embeber_lote(textos)
        filas = [{"eid": eid, "embedding": vec} for (eid, _), vec in zip(lote, vectores)]
        escribir_vectores(driver, filas)
        hechos += len(filas)
        print(f"  {hechos}/{len(preparados)} embebidos y guardados")
        time.sleep(0.2)
verificar(driver)
print("Backfill listo.")

Indices creados/verificados.
Articulos pendientes de embeber: 0
Verificacion: {'total': 1470, 'con_embedding': 1442, 'pendientes': 0}
Backfill listo.


## Parte B — Retriever GraphRAG (uso normal)

### Retriever + pipeline

Vector sobre `recuperable_embedding` → expansión Cypher a la norma, su vigencia y `DEROGA`/`MODIFICA`/`REGLAMENTA`. Construye `retriever` y el pipeline `rag`.

In [7]:
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from neo4j_graphrag.generation import GraphRAG

RETRIEVAL_QUERY = """
OPTIONAL MATCH (norma)-[:TIENE_ARTICULO]->(node)
    WHERE norma:Ley OR norma:Decreto OR norma:Resolucion
WITH node, collect(norma)[0] AS norma
RETURN
    node.texto      AS texto,
    node.numero     AS articulo,
    node.ley_numero AS norma_numero,
    CASE WHEN norma IS NULL THEN null
         ELSE head([l IN labels(norma) WHERE l IN ['Ley','Decreto','Resolucion']]) END AS norma_tipo,
    CASE WHEN norma IS NULL THEN null ELSE norma.nombre_completo END AS norma_titulo,
    CASE WHEN norma IS NULL THEN 'sin_dato' ELSE coalesce(norma.estado,'sin_dato') END AS estado,
    CASE WHEN norma IS NULL THEN null ELSE norma.derogada_por END AS derogada_por,
    CASE WHEN norma IS NULL THEN null ELSE norma.vigente_hasta END AS vigente_hasta,
    CASE WHEN norma IS NULL THEN null ELSE norma.fuente_url END AS fuente_url,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:DEROGA]-(x)     | toString(x.numero) ] END AS derogada_por_rel,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:MODIFICA]-(y)   | toString(y.numero) ] END AS modificada_por_rel,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:REGLAMENTA]-(d) | toString(d.numero) + ' — ' + coalesce(d.nombre_completo,'') ] END AS reglamentada_por,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)-[:REGLAMENTA]->(l) | toString(l.numero) + ' — ' + coalesce(l.nombre_completo,'') ] END AS reglamenta_a
"""

def formatear(record):
    estado = record.get("estado")
    if record.get("derogada_por"):
        marca = f"  [DEROGADA por {record.get('derogada_por')}]"
    elif record.get("vigente_hasta"):
        marca = f"  [vigente hasta {record.get('vigente_hasta')}]"
    elif estado and str(estado).lower() not in ("vigente", "sin_dato"):
        marca = f"  [estado: {estado}]"
    else:
        marca = ""
    tipo = record.get("norma_tipo") or ""
    num  = record.get("norma_numero") or ""
    encabezado = f"{tipo} {num}, Artículo {record.get('articulo')}{marca}".strip()

    extras = []
    if record.get("reglamentada_por"):
        extras.append("Reglamentada por: " + "; ".join(record.get("reglamentada_por")))
    if record.get("reglamenta_a"):
        extras.append("Reglamenta a: " + "; ".join(record.get("reglamenta_a")))
    if record.get("modificada_por_rel"):
        extras.append("Modificada por: " + ", ".join(record.get("modificada_por_rel")))
    extra_txt = ("\n" + " | ".join(extras)) if extras else ""

    return RetrieverResultItem(
        content=f"{encabezado}\n{record.get('texto') or ''}{extra_txt}",
        metadata={
            "norma": num,
            "articulo": record.get("articulo"),
            "estado": estado,
            "derogada_por": record.get("derogada_por"),
            "fuente_url": record.get("fuente_url"),
        },
    )

retriever = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX_NAME,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=formatear,
    neo4j_database=NEO4J_DATABASE,
)
rag = GraphRAG(retriever=retriever, llm=llm, prompt_template=prompt_template)
print("Retriever + pipeline GraphRAG listos.")

Retriever + pipeline GraphRAG listos.


### Probar solo el retriever

Sin LLM (barato). Deberías ver el encabezado, la marca de vigencia si corresponde, y las líneas de reglamenta/modifica cuando existan.

In [19]:
# Probar SOLO el retriever (sin LLM): ver que contexto trae
pregunta = "¿Qué beneficios ofrece el régimen de zonas francas?"
res = retriever.search(query_text=pregunta, top_k=TOP_K)
for i, it in enumerate(res.items, 1):
    print(f"[{i}] {it.metadata}")
    print(it.content)
    print("---")

[1] {'norma': '5102', 'articulo': 1, 'estado': 'derogada', 'derogada_por': '7452/25', 'fuente_url': 'https://baselegal.com.py/docs/542a079e-bfcb-11ea-b286-525400c761ca'}
Ley 5102, Artículo 1  [DEROGADA por 7452/25]
La presente Ley tiene por objeto establecer normas y mecanismos para promover, a través de la participación público-privada, las inversiones en infraestructura pública y en la prestación de los servicios a que las mismas estén destinadas o que sean comple
---
[2] {'norma': '5102', 'articulo': 5, 'estado': 'derogada', 'derogada_por': '7452/25', 'fuente_url': 'https://baselegal.com.py/docs/542a079e-bfcb-11ea-b286-525400c761ca'}
Ley 5102, Artículo 5  [DEROGADA por 7452/25]
Los contratos de participación público-privada se regirán por los términos y condiciones del contrato, las disposiciones de la presente Ley y por la reglamentación que dicte el Poder Ejecutivo y por las demás disposiciones legales en cuanto fueran aplicab
---
[3] {'norma': '5102', 'articulo': 4, 'estado': 'de

### Demo del pipeline (retriever + gpt-4o)

Respuesta anclada al contexto, citando norma/artículo y respetando la vigencia.

In [9]:
# Demo del pipeline GraphRAG completo (usa 'rag' del bloque del retriever)
pregunta = "¿Qué beneficios ofrece el régimen de zonas francas?"
resp = rag.search(query_text=pregunta, retriever_config={"top_k": TOP_K}, return_context=True)
print(resp.answer)

El régimen de Zonas Francas en Paraguay ofrece varios beneficios, principalmente en el ámbito tributario. Según la Ley 523/95, Artículo 13, las actividades realizadas en Zonas Francas y los resultados obtenidos por los Usuarios están exentos de todo tributo nacional, departamental o municipal, con excepción del régimen tributario contemplado dentro del mismo capítulo de la ley. Además, la exoneración tributaria se extiende a la constitución de las sociedades Usuarias de las Zonas Francas y a las remesas de utilidades o dividendos a terceros países. También incluye la exención tributaria por el pago de regalías, comisiones, honorarios, intereses y otras remuneraciones por servicios, asistencia técnica, transferencia de tecnología, préstamos y financiamiento, alquiler de equipos y otros servicios prestados desde terceros países a los Usuarios de Zonas Francas.


## Parte C — Baseline vector-only y comparación

El **baseline** es el mismo sistema con el **grafo apagado**: idéntico embedder, LLM, prompt y `top_k`, pero el `retrieval_query` **no expande** al grafo. Cualquier diferencia es atribuible al grafo.

In [10]:
# Baseline vector-only: MISMO todo (embedder, LLM, prompt, top_k), pero el
# retrieval_query NO expande al grafo (solo devuelve el articulo).
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from neo4j_graphrag.generation import GraphRAG

RETRIEVAL_QUERY_BASELINE = """
RETURN node.texto AS texto, node.numero AS articulo, node.ley_numero AS norma_numero
"""

def formatear_baseline(record):
    num = record.get("norma_numero") or ""
    art = record.get("articulo")
    encabezado = f"Norma {num}, Artículo {art}".strip()
    return RetrieverResultItem(
        content=f"{encabezado}\n{record.get('texto') or ''}",
        metadata={"norma": num, "articulo": art},
    )

retriever_baseline = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX_NAME,
    retrieval_query=RETRIEVAL_QUERY_BASELINE,
    embedder=embedder,
    result_formatter=formatear_baseline,
    neo4j_database=NEO4J_DATABASE,
)
rag_baseline = GraphRAG(retriever=retriever_baseline, llm=llm, prompt_template=prompt_template)
print("Baseline + pipeline baseline listos.")

Baseline + pipeline baseline listos.


### Comparar baseline vs GraphRAG

Misma pregunta por los dos. En la consulta testigo de la **Ley 60/90 (derogada)**, el baseline tiende a listarla como vigente (la falla), y el GraphRAG avisa la derogación (el acierto).

In [11]:
# Comparar baseline vs GraphRAG sobre la MISMA pregunta (aisla el aporte del grafo).
# Chequeo de dependencias: si algo falta, te dice qué celda correr (en vez de un NameError).
_faltan = [n for n in ("TOP_K", "rag", "rag_baseline") if n not in globals()]
if _faltan:
    print("⚠ Faltan objetos para correr esta celda:", ", ".join(_faltan))
    if "TOP_K" in _faltan:
        print("   → Corré el SETUP (celda de config): define TOP_K y la conexión.")
    if "rag" in _faltan:
        print("   → Corré la celda del RETRIEVER (Parte B): crea 'retriever' y 'rag' "
              "(termina en 'Retriever + pipeline GraphRAG listos.').")
    if "rag_baseline" in _faltan:
        print("   → Corré la celda del BASELINE (Parte C): crea 'rag_baseline' "
              "(termina en 'Baseline + pipeline baseline listos.').")
    print("   Atajo: Runtime → 'Run before' parado en esta celda corre todo lo de arriba en orden.")
else:
    pregunta = "¿qué incentivos fiscales ofrece la Ley 60/90?"

    print("=== BASELINE (vector-only, grafo apagado) ===")
    print(rag_baseline.search(query_text=pregunta, retriever_config={"top_k": TOP_K}).answer)

    print("\n=== GraphRAG (con grafo) ===")
    print(rag.search(query_text=pregunta, retriever_config={"top_k": TOP_K}).answer)

=== BASELINE (vector-only, grafo apagado) ===


La Ley N° 60/90 ofrece incentivos fiscales con el objetivo de promover e incrementar las inversiones de capital de origen nacional y/o extranjero. Los beneficios fiscales se otorgan a personas físicas y jurídicas radicadas en Paraguay cuyas inversiones se alineen con la política económica y social del Gobierno Nacional. Los objetivos específicos para otorgar estos beneficios incluyen:

a) El acrecentamiento de la producción de bienes y servicios.
b) La creación de fuentes de trabajo permanente.
c) El fomento de las exportaciones y la sustitución de importaciones.
d) La incorporación de tecnologías que aumenten la eficiencia productiva y permitan una mejor utilización de materias primas, mano de obra y recursos energéticos nacionales.
e) La inversión y reinversión de utilidades en bienes de capital.

Estos incentivos están detallados en el Artículo 1 de la Ley N° 60/90.

=== GraphRAG (con grafo) ===


La Ley 60/90 ha sido derogada por la Ley 7548/25, por lo que no está vigente. Por lo tanto, no puedo presentar los incentivos fiscales de la Ley 60/90 como vigentes. Si necesitas información sobre incentivos fiscales actuales, te recomiendo revisar las disposiciones de la Ley 7548/25 y sus artículos correspondientes.


## Parte D — Agente conversacional (LangGraph · Etapas 1-2-3)

**Requisitos:** haber corrido el **Setup** y la celda del **retriever (Parte B)** — usa `retriever`, `llm` y `TOP_K`.

Grafo: **reescribir** (Etapa 2: consulta autónoma con el historial) → **recuperar** → **graduar** (Etapa 3: el LLM juzga si el contexto alcanza) → según el veredicto: **responder** (genera con memoria, Etapa 1), **reformular** → volver a **recuperar** (un reintento), o **sin_contexto** (respuesta honesta si no hay info). Checkpointer con `thread_id` para la memoria. Solo `langgraph`.

La generación (`responder`) usa `retriever` + `llm` por separado (no `rag.search`) para poder graduar el contexto antes de responder; la vigencia sigue expuesta porque el contexto trae las marcas `[DEROGADA…]` del retriever.

### Grafo (reescribir → recuperar → graduar → responder / reformular / sin_contexto)

In [20]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM = (
    "Sos un asistente legal sobre derecho de inversiones de Paraguay. "
    "Respondé usando SOLO el contexto provisto. Citá la norma y el artículo en cada afirmación. "
    "Si una norma figura como DEROGADA o con estado distinto de vigente, aclaralo y NO la "
    "presentes como vigente. Si el contexto no alcanza, decilo."
)

class EstadoConv(TypedDict):
    messages: Annotated[list, operator.add]
    consulta: str      # consulta (reescrita) usada para buscar
    context: str       # contexto recuperado
    intentos: int      # cuantas veces se busco (corta el bucle)
    relevante: bool    # veredicto del grading

# --- Etapa 2: reescritura history-aware ---
def reescribir(state: EstadoConv):
    if len(state["messages"]) <= 1:
        return {"consulta": state["messages"][-1]["content"], "intentos": 0}
    historial = "\n".join(f'{m["role"]}: {m["content"]}' for m in state["messages"][:-1])
    pregunta = state["messages"][-1]["content"]
    prompt = (
        "Dada la conversación previa y una pregunta de seguimiento, reescribí la pregunta como "
        "una consulta AUTÓNOMA que se entienda sin el historial (resolvé 'esa ley', 'ese régimen', "
        "'ahí'). Si ya es autónoma, devolvela igual. Devolvé SOLO la consulta, sin comillas.\n\n"
        f"# Conversación:\n{historial}\n\n# Pregunta:\n{pregunta}\n\n# Consulta autónoma:"
    )
    return {"consulta": llm.invoke(prompt).content.strip(), "intentos": 0}

def recuperar(state: EstadoConv):
    res = retriever.search(query_text=state["consulta"], top_k=TOP_K)
    context = "\n\n".join(it.content for it in res.items)
    return {"context": context, "intentos": state.get("intentos", 0) + 1}

# --- Etapa 3: grading + re-busqueda (corrective RAG) ---
def graduar(state: EstadoConv):
    prompt = (
        "Decidí si el CONTEXTO alcanza para responder la PREGUNTA de forma fundamentada. "
        "Respondé SOLO con 'SI' o 'NO'.\n\n"
        "IMPORTANTE: si el contexto contiene artículos de la norma consultada, alcanza — "
        "AUNQUE la norma figure como DEROGADA, modificada o no vigente. Advertir que una norma "
        "está derogada (y por cuál fue reemplazada) es una respuesta VÁLIDA y útil, no una falta "
        "de información. Respondé 'NO' solo si el contexto es de otra materia o no tiene nada "
        "que ver con la pregunta.\n\n"
        f"# Pregunta:\n{state['consulta']}\n\n# Contexto:\n{state['context'][:4000]}\n\n# ¿Alcanza? (SI/NO):"
    )
    veredicto = llm.invoke(prompt).content.strip().upper()
    return {"relevante": veredicto.startswith("SI")}

def decidir(state: EstadoConv):
    # Router: SOLO lee el estado y devuelve el nombre del proximo nodo (sin llamar al LLM aca)
    if state["relevante"]:
        return "responder"
    if state["intentos"] < 2:     # un reintento con reformulacion
        return "reformular"
    return "sin_contexto"

def reformular(state: EstadoConv):
    prompt = (
        "La búsqueda anterior no trajo contexto suficiente. Reformulá la consulta con otros "
        "términos o de forma más general para mejorar la recuperación. Devolvé SOLO la nueva consulta.\n\n"
        f"# Consulta anterior:\n{state['consulta']}"
    )
    return {"consulta": llm.invoke(prompt).content.strip()}

def responder(state: EstadoConv):
    history = state["messages"][:-1]
    resp = llm.invoke(
        input=f"# Contexto:\n{state['context']}\n\n# Pregunta:\n{state['consulta']}",
        message_history=history,
        system_instruction=SYSTEM,
    )
    return {"messages": [{"role": "assistant", "content": resp.content}]}

def sin_contexto(state: EstadoConv):
    msg = ("No encontré en la base normativa cargada información suficiente para responder eso "
           "con fundamento. ¿Podés reformular la pregunta o dar más detalle?")
    return {"messages": [{"role": "assistant", "content": msg}]}

builder = StateGraph(EstadoConv)
builder.add_node("reescribir", reescribir)
builder.add_node("recuperar", recuperar)
builder.add_node("graduar", graduar)
builder.add_node("reformular", reformular)
builder.add_node("responder", responder)
builder.add_node("sin_contexto", sin_contexto)

builder.add_edge(START, "reescribir")
builder.add_edge("reescribir", "recuperar")
builder.add_edge("recuperar", "graduar")
builder.add_conditional_edges("graduar", decidir, {
    "responder": "responder",
    "reformular": "reformular",
    "sin_contexto": "sin_contexto",
})
builder.add_edge("reformular", "recuperar")   # bucle: vuelve a buscar con la consulta reformulada
builder.add_edge("responder", END)
builder.add_edge("sin_contexto", END)

chat_graph = builder.compile(checkpointer=InMemorySaver())
print("Grafo conversacional (memoria + reescritura + corrective RAG) listo.")

Grafo conversacional (memoria + reescritura + corrective RAG) listo.


### Demo

Se imprime la **consulta usada** y cuántas **búsquedas** hizo. Muestra los tres caminos: conversación normal (memoria + reescritura), y una pregunta fuera de dominio que dispara el grading → reformulación → respuesta honesta de que no hay info suficiente.

In [22]:
import uuid

def preguntar(chat_id, texto):
    out = chat_graph.invoke({"messages": [{"role": "user", "content": texto}]},
                            {"configurable": {"thread_id": chat_id}})
    print("Usuario:", texto)
    print("  (consulta usada:", out["consulta"], "| busquedas:", out["intentos"], ")")
    print("Bot:", out["messages"][-1]["content"], "\n")

# Conversación normal: memoria (Etapa 1) + reescritura (Etapa 2)
chat = str(uuid.uuid4())
preguntar(chat, "¿Qué es el régimen de zonas francas?")
preguntar(chat, "¿Qué actividades permite?")

# Fuera de dominio (otra conversación): grading NO -> reformula -> fallback honesto (Etapa 3)
preguntar(str(uuid.uuid4()), "¿Cuál es la mejor receta de sopa paraguaya?")

#Pregunta ley nueva
preguntar(chat, "¿qué establece la Ley 5102 sobre alianza público-privada?")

Usuario: ¿Qué es el régimen de zonas francas?
  (consulta usada: ¿Qué es el régimen de zonas francas? | busquedas: 1 )
Bot: El régimen de zonas francas en Paraguay, según la Ley 523/95, se refiere a espacios del territorio nacional que son localizados y autorizados por el Poder Ejecutivo. Estas zonas están sujetas a un control fiscal, aduanero y administrativo específico establecido por la ley y sus reglamentaciones (Artículo 1). Las zonas francas deben instalarse en áreas de propiedad privada y estar cercadas para garantizar su aislamiento del Territorio Aduanero, teniendo un único sector de entrada y salida (Artículo 2). Dentro de estas zonas, se pueden desarrollar actividades comerciales, industriales y de servicios, con ciertas especificaciones y condiciones (Artículo 3). Además, el Consejo Nacional de Zonas Francas es el organismo encargado de la fiscalización y control de estas zonas (Artículo 37). 



Usuario: ¿Qué actividades permite?
  (consulta usada: ¿Qué actividades permite el régimen de zonas francas en Paraguay según la Ley 523/95? | busquedas: 1 )
Bot: Según la Ley 523/95, el régimen de zonas francas en Paraguay permite desarrollar las siguientes actividades (Artículo 3):

a) **Comerciales**: Actividades en las cuales los usuarios se dedican a la internación de bienes destinados para su intermediación sin que sufran transformación o modificación. Esto incluye el depósito, selección, clasificación, manipulación, y mezcla de mercaderías o materias primas.

b) **Industriales**: Actividades en las cuales los usuarios se dedican a la fabricación de bienes destinados a la exportación al exterior, mediante la transformación de materias primas y/o productos semielaborados de origen nacional o importado. Esto incluye actividades clasificadas como ensamblaje.

c) **Servicios**: Actividades en las cuales los usuarios se dedican a reparaciones y mantenimiento de equipos y maquinarias. A

Usuario: ¿Cuál es la mejor receta de sopa paraguaya?
  (consulta usada: Receta tradicional de sopa paraguaya | busquedas: 2 )
Bot: No encontré en la base normativa cargada información suficiente para responder eso con fundamento. ¿Podés reformular la pregunta o dar más detalle? 



Usuario: ¿qué establece la Ley 5102 sobre alianza público-privada?
  (consulta usada: Ley 5102 de Paraguay sobre colaboración entre sector público y privado | busquedas: 2 )
Bot: La Ley 5102 de Paraguay, que regulaba la colaboración entre el sector público y privado, ha sido derogada por la Ley 7452/25. Por lo tanto, no está vigente y no debe ser considerada como parte del marco legal actual para la colaboración público-privada en Paraguay. Si necesitas información sobre el régimen actual, sería necesario consultar la legislación vigente que reemplazó a la Ley 5102. 



## Notas

- `TOP_K` = cuántos artículos recupera (no la ventana de contexto); más alto = más contexto y costo.
- Warning `db.index.vector.queryNodes` deprecado: inofensivo. Descomentá la línea de `logging` en la config para silenciarlo.
- Etapas siguientes de LangGraph: reescritura de consulta (Etapa 2) y grading + re-búsqueda (Etapa 3).
- Al terminar, `driver.close()` si no vas a seguir usando la conexión.